# Keyword Grouping Pipeline
Groups similar/duplicate keywords from your `aggregate.json` using:
- **Step 1** — Rule-based (instant, no model needed)
- **Step 2** — Semantic grouping via Ollama (local LLM, free)
- **Step 3** — Merge and save output

### Requirements
1. Ollama must be running: open a terminal and run `ollama serve`
2. Model must be pulled: `ollama pull mistral`
3. Put `aggregate.json` in the same folder as this notebook

---
## Cell 1 — Configuration
**Only edit this cell.** Change the model name and file paths here.

In [22]:
# ── EDIT THESE ────────────────────────────────────────────────
INPUT_FILE   = "aggregate.json"        # your input file
OUTPUT_FILE  = "keywords_grouped.json" # output will be saved here
LOG_FILE     = "grouping_log.txt"      # human-readable log of all groups

# Model to use — you have 8GB RAM + RTX 3050 4GB VRAM, so mistral works great
# Options: "mistral"  (best quality, recommended)
#          "llama3.2" (also good)
#          "llama3.2:1b" (fastest, less accurate)
OLLAMA_MODEL = "llama3.2"

OLLAMA_URL          = "http://localhost:11434/api/generate"
SEMANTIC_BATCH_SIZE = 150   # keywords per batch — 150 works well for mistral
RETRY_DELAY         = 3
# ──────────────────────────────────────────────────────────────

print(f"Config loaded. Model: {OLLAMA_MODEL}, Batch size: {SEMANTIC_BATCH_SIZE}")

Config loaded. Model: llama3.2, Batch size: 150


---
## Cell 2 — Imports and Helper Functions

In [23]:
import json
import re
import unicodedata
import urllib.request
import urllib.error
import time
from collections import defaultdict

# install tqdm for progress bars if not present
try:
    from tqdm.notebook import tqdm
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm", "-q"])
    from tqdm.notebook import tqdm

# ── Abbreviation map ──────────────────────────────────────────
ABBREV_MAP = {
    "co2":   "carbon dioxide",
    "h2o":   "water",
    "h2s":   "hydrogen sulfide",
    "ch4":   "methane",
    "n2":    "nitrogen",
    "o2":    "oxygen",
    "h2":    "hydrogen",
    "nox":   "nitrogen oxides",
    "sox":   "sulfur oxides",
    "vle":   "vapor-liquid equilibrium",
    "lle":   "liquid-liquid equilibrium",
    "sle":   "solid-liquid equilibrium",
    "md":    "molecular dynamics",
    "mc":    "monte carlo",
    "dft":   "density functional theory",
    "ml":    "machine learning",
    "dl":    "deep learning",
    "nn":    "neural network",
    "ai":    "artificial intelligence",
    "cfd":   "computational fluid dynamics",
    "pde":   "partial differential equation",
    "ode":   "ordinary differential equation",
    "il":    "ionic liquid",
    "ils":   "ionic liquid",
    "mea":   "monoethanolamine",
    "mdea":  "methyldiethanolamine",
    "pz":    "piperazine",
    "dea":   "diethanolamine",
    "ccs":   "carbon capture and storage",
    "ccus":  "carbon capture utilization and storage",
    "tga":   "thermogravimetric analysis",
    "dsc":   "differential scanning calorimetry",
    "nmr":   "nuclear magnetic resonance",
    "ftir":  "fourier transform infrared spectroscopy",
    "gc":    "gas chromatography",
    "hplc":  "high performance liquid chromatography",
    "pvt":   "pressure volume temperature",
    "eos":   "equation of state",
    "saft":  "statistical associating fluid theory",
    "pr":    "peng-robinson",
    "srk":   "soave-redlich-kwong",
    "mof":   "metal-organic framework",
    "cof":   "covalent organic framework",
    "htc":   "hydrothermal carbonization",
    "htl":   "hydrothermal liquefaction",
    "cstr":  "continuous stirred tank reactor",
    "pfr":   "plug flow reactor",
    "mpc":   "model predictive control",
    "pid":   "proportional integral derivative",
    "lca":   "life cycle assessment",
    "tac":   "total annualized cost",
    "capex": "capital expenditure",
    "opex":  "operational expenditure",
    "ro":    "reverse osmosis",
    "mf":    "microfiltration",
    "uf":    "ultrafiltration",
    "nf":    "nanofiltration",
    "ed":    "electrodialysis",
    "ghg":   "greenhouse gas",
}

# ── Helper functions ──────────────────────────────────────────
def normalize(text):
    text = text.lower().strip()
    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r"[\u2010-\u2015\u2212\ufe58\ufe63\uff0d]", "-", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"-+", "-", text)
    return text

def depluralize(text):
    if text.endswith("ies") and len(text) > 4:
        return text[:-3] + "y"
    if text.endswith("ses") and len(text) > 4:
        return text[:-2]
    if text.endswith("s") and not text.endswith("ss") and len(text) > 3:
        return text[:-1]
    return text

def rule_based_canonical(kw):
    norm = normalize(kw)
    words = norm.split()
    expanded = [ABBREV_MAP.get(w, w) for w in words]
    norm = " ".join(expanded)
    parts = norm.split()
    if parts:
        parts[-1] = depluralize(parts[-1])
    return " ".join(parts)

def merge_keyword_data(base, extra):
    base["total_count"] = base.get("total_count", 0) + extra.get("total_count", 0)
    for year, cnt in extra.get("years", {}).items():
        base.setdefault("years", {})[year] = base["years"].get(year, 0) + cnt
    for country, cnt in extra.get("countries", {}).items():
        base.setdefault("countries", {})[country] = base["countries"].get(country, 0) + cnt
    for author, cnt in extra.get("authors", {}).items():
        base.setdefault("authors", {})[author] = base["authors"].get(author, 0) + cnt
    for year, papers in extra.get("papers_by_year", {}).items():
        existing = base.setdefault("papers_by_year", {}).setdefault(year, [])
        seen = {(p["title"], p.get("source_file", "")) for p in existing}
        for p in papers:
            key = (p["title"], p.get("source_file", ""))
            if key not in seen:
                existing.append(p)
                seen.add(key)
    return base

print("Helper functions loaded.")

Helper functions loaded.


---
## Cell 3 — Load Data

In [24]:
print(f"Loading {INPUT_FILE}...")
with open(INPUT_FILE, encoding="utf-8") as f:
    data = json.load(f)

print(f"Total keywords loaded: {len(data):,}")
print(f"Sample keywords: {list(data.keys())[:5]}")

Loading aggregate.json...
Total keywords loaded: 19,592
Sample keywords: ['vapor-liquid equilibrium', 'acid gas absorption', 'mdea‑piperazine blends', 'opls‑aa force field', 'accurate point charges']


---
## Cell 4 — Step 1: Rule-Based Grouping
Fast, no model needed. Catches unicode variants, plurals, known abbreviations.

In [25]:
print("Step 1: Rule-based grouping...")

norm_to_originals = defaultdict(list)
for kw in tqdm(data.keys(), desc="Normalizing"):
    norm = rule_based_canonical(kw)
    norm_to_originals[norm].append(kw)

rule_groups   = {}  # canonical -> [aliases]
ungrouped     = []  # keywords with no rule-match

for norm, originals in norm_to_originals.items():
    if len(originals) == 1:
        ungrouped.append(originals[0])
    else:
        canonical = max(originals, key=lambda k: data[k].get("total_count", 0))
        aliases   = [k for k in originals if k != canonical]
        rule_groups[canonical] = aliases

print(f"\nRule-based results:")
print(f"  Groups found       : {len(rule_groups):,}")
print(f"  Keywords merged    : {sum(len(v) for v in rule_groups.values()):,}")
print(f"  Still ungrouped    : {len(ungrouped):,}")

print("\nSample rule-based groups:")
for canon, aliases in list(rule_groups.items())[:8]:
    print(f"  [{canon}]  <-  {aliases}")

Step 1: Rule-based grouping...


Normalizing:   0%|          | 0/19592 [00:00<?, ?it/s]


Rule-based results:
  Groups found       : 392
  Keywords merged    : 404
  Still ungrouped    : 18,796

Sample rule-based groups:
  [vapor-liquid equilibrium]  <-  ['vapor‑liquid equilibrium']
  [redlich-kister correlation]  <-  ['redlich‑kister correlation']
  [vibrating-tube densimetry]  <-  ['vibrating‑tube densimetry']
  [computational fluid dynamics]  <-  ['cfd']
  [cation exchange membrane]  <-  ['cation exchange membranes']
  [monte carlo simulations]  <-  ['monte carlo simulation']
  [techno-economic analysis]  <-  ['techno‑economic analysis']
  [mixed-integer nonlinear programming]  <-  ['mixed‑integer nonlinear programming']


---
## Cell 5 — Check Ollama is Running
Run this before Step 2. If it fails, open a terminal and run `ollama serve`.

In [26]:
import urllib.request

try:
    with urllib.request.urlopen("http://localhost:11434", timeout=5) as r:
        print(f"Ollama is running! Status: {r.status}")
except Exception as e:
    print(f"Ollama NOT reachable: {e}")
    print("\nFix: open a new terminal and run:  ollama serve")
    print(f"Then pull the model:               ollama pull {OLLAMA_MODEL}")

Ollama is running! Status: 200


---
## Cell 6 — Step 2: Semantic Grouping via Ollama
Sends batches of 150 keywords to your local model. With RTX 3050 + mistral expect ~30-45 sec/batch.

**This will take 1-2 hours for 19k keywords. You can leave it running.**

Progress is saved as it goes — if interrupted, partial results are still used in Step 3.

In [27]:
import ollama

response = ollama.chat(
    model='llama3',  # make sure you have this model pulled
    messages=[
        {"role": "user", "content": "Explain recursion in one sentence"}
    ]
)

print(response['message']['content'])

Recursion is a programming technique where a function calls itself repeatedly until it reaches a base case, allowing the function to solve problems that would be difficult or impossible to solve using traditional iterative methods.


In [28]:
import json
import urllib.request
import urllib.error
import time
import re
from tqdm import tqdm

SEMANTIC_BATCH_SIZE = 130

SEMANTIC_PROMPT = """You are a research keyword deduplication expert for chemical engineering and related sciences.

Below is a numbered list of research keywords. Identify groups of keywords that refer to the SAME or very similar concept.

BE CONSERVATIVE — only group keywords that clearly mean the same thing.

Keywords:
{keyword_list}

Return ONLY a JSON array.

STRICT RULES:
- Each element MUST be an object with keys "canonical" and "members"
- "members" must be a list of strings
- Include canonical inside members
- Only include groups with 2+ members
- DO NOT return list of lists
- DO NOT explain anything
"""

MISSED_KEYWORDS_FILE = "missed_keywords.txt"


def call_ollama(prompt, retries=3):
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.0, "num_predict": 2048}
    }).encode()

    req = urllib.request.Request(
        OLLAMA_URL,
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=300) as resp:
                return json.loads(resp.read()).get("response", "")
        except urllib.error.URLError as e:
            if attempt == 0:
                print("  ⚠️  Cannot connect to Ollama — is 'ollama serve' running?")
            if attempt < retries - 1:
                time.sleep(RETRY_DELAY)
            else:
                raise RuntimeError(f"Ollama not reachable: {e}")
        except Exception:
            if attempt < retries - 1:
                time.sleep(RETRY_DELAY)
            else:
                raise


# ── Run semantic grouping ──────────────────────────────────────────────────────

semantic_groups  = []
missed_keywords  = []   # keywords from failed/unparseable batches
batch_log        = []   # per-batch summary rows for final table

total_batches = (len(ungrouped) + SEMANTIC_BATCH_SIZE - 1) // SEMANTIC_BATCH_SIZE

print("=" * 60)
print(f"  Semantic Grouping")
print(f"  Keywords : {len(ungrouped):,}")
print(f"  Model    : {OLLAMA_MODEL}")
print(f"  Batches  : {total_batches}  (size {SEMANTIC_BATCH_SIZE})")
print("=" * 60)

pbar = tqdm(
    total=total_batches,
    desc="  Progress",
    unit="batch",
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]"
)

for batch_num in range(total_batches):
    start = batch_num * SEMANTIC_BATCH_SIZE
    batch = ungrouped[start : start + SEMANTIC_BATCH_SIZE]

    kw_list = "\n".join(f"{i+1}. {kw}" for i, kw in enumerate(batch))
    prompt  = SEMANTIC_PROMPT.format(keyword_list=kw_list)

    status        = "ok"
    groups_found  = 0
    t0            = time.time()

    try:
        text    = call_ollama(prompt)
        elapsed = time.time() - t0

        # Extract JSON array from response
        match = re.search(r"\[.*\]", text, re.DOTALL)
        if match:
            text = match.group(0)

        # Strip markdown fences if present
        text = re.sub(r"^```[a-z]*\n?", "", text.strip())
        text = re.sub(r"\n?```$",        "", text).strip()

        if not text or text == "[]":
            status = "no_groups"
        else:
            try:
                groups_raw = json.loads(text)
            except json.JSONDecodeError:
                # ── MISSED BATCH: save keywords instead of discarding ──
                missed_keywords.extend(batch)
                status = "parse_error"
                batch_log.append((batch_num + 1, len(batch), 0, f"{elapsed:.1f}s", status))
                pbar.set_postfix({"groups": len(semantic_groups), "missed": len(missed_keywords)})
                pbar.update(1)
                time.sleep(0.2)
                continue

            batch_lower = {k.lower().strip(): k for k in batch}

            for g in groups_raw:
                if isinstance(g, list):
                    members, canonical = g, (g[0] if g else "")
                elif isinstance(g, dict):
                    members   = g.get("members", [])
                    canonical = g.get("canonical", "")
                else:
                    continue

                members_original = [
                    batch_lower[str(m).lower().strip()]
                    for m in members
                    if str(m).lower().strip() in batch_lower
                ]

                if len(members_original) >= 2:
                    canon_orig = batch_lower.get(str(canonical).lower().strip()) or members_original[0]
                    semantic_groups.append({"canonical": canon_orig, "members": members_original})
                    groups_found += 1

    except Exception as e:
        elapsed = time.time() - t0
        # ── MISSED BATCH: network / timeout failure ──
        missed_keywords.extend(batch)
        status = f"error"
        tqdm.write(f"  ✗ Batch {batch_num+1:>3} failed after {elapsed:.1f}s — {e}")
        batch_log.append((batch_num + 1, len(batch), 0, f"{elapsed:.1f}s", status))
        pbar.set_postfix({"groups": len(semantic_groups), "missed": len(missed_keywords)})
        pbar.update(1)
        time.sleep(0.2)
        continue

    batch_log.append((batch_num + 1, len(batch), groups_found, f"{elapsed:.1f}s", status))
    pbar.set_postfix({"groups": len(semantic_groups), "missed": len(missed_keywords)})
    pbar.update(1)
    time.sleep(0.2)

pbar.close()

# ── Save missed keywords to file ───────────────────────────────────────────────
if missed_keywords:
    with open(MISSED_KEYWORDS_FILE, "w", encoding="utf-8") as f:
        for kw in missed_keywords:
            f.write(kw + "\n")

# ── Final summary table ────────────────────────────────────────────────────────
ok_batches      = sum(1 for r in batch_log if r[4] in ("ok", "no_groups"))
errored_batches = sum(1 for r in batch_log if r[4] not in ("ok", "no_groups"))

print("\n" + "=" * 60)
print("  Run Summary")
print("=" * 60)
print(f"  {'Metric':<30} {'Value':>10}")
print(f"  {'-'*40}")
print(f"  {'Total batches':<30} {total_batches:>10,}")
print(f"  {'Successful batches':<30} {ok_batches:>10,}")
print(f"  {'Failed batches':<30} {errored_batches:>10,}")
print(f"  {'Semantic groups found':<30} {len(semantic_groups):>10,}")
print(f"  {'Keywords missed (saved)':<30} {len(missed_keywords):>10,}")
if missed_keywords:
    print(f"  {'Missed saved to':<30} {MISSED_KEYWORDS_FILE:>10}")
print("=" * 60)

# ── Per-batch detail (only failures, to keep output clean) ─────────────────────
failures = [(b, kw, g, t, s) for b, kw, g, t, s in batch_log if s not in ("ok", "no_groups")]
if failures:
    print("\n  Failed batch details:")
    print(f"  {'Batch':>6}  {'Keywords':>8}  {'Time':>7}  Status")
    print(f"  {'─'*40}")
    for b, kw, g, t, s in failures:
        print(f"  {b:>6}  {kw:>8}  {t:>7}  {s}")
    print()


  Semantic Grouping
  Keywords : 18,796
  Model    : llama3.2
  Batches  : 145  (size 130)


Semantic batches:  79%|███████▊  | 114/145 [2:35:46<42:21, 81.99s/batch, total_groups=1173, errors=41, this_batch=36]


KeyboardInterrupt: 

---
## Cell 7 — Step 3: Merge and Build Output

In [ ]:
print("Step 3: Merging all groups and building output...")

# Collect all merges
all_merges = {}
for canonical, aliases in rule_groups.items():
    all_merges[canonical] = list(aliases)
for g in semantic_groups:
    canonical = g["canonical"]
    aliases   = [m for m in g["members"] if m != canonical]
    if canonical in all_merges:
        all_merges[canonical].extend(aliases)
    else:
        all_merges[canonical] = aliases

all_aliases = set()
for aliases in all_merges.values():
    all_aliases.update(aliases)

# Build result
result = {}
for kw, kw_data in tqdm(data.items(), desc="Building output"):
    if kw in all_aliases:
        continue  # absorbed into its canonical
    entry = json.loads(json.dumps(kw_data))
    if kw in all_merges:
        for alias in all_merges[kw]:
            if alias in data:
                entry = merge_keyword_data(entry, data[alias])
        entry["canonical_keyword"] = kw
        entry["grouped_with"]      = all_merges[kw]
    else:
        entry["canonical_keyword"] = kw
        entry["grouped_with"]      = []
    result[kw] = entry

total_groups  = sum(1 for v in result.values() if v["grouped_with"])
total_aliases = sum(len(v["grouped_with"]) for v in result.values())

print(f"\nResults:")
print(f"  Original keywords  : {len(data):,}")
print(f"  Output keywords    : {len(result):,}")
print(f"  Keywords merged    : {total_aliases:,}")
print(f"  Groups formed      : {total_groups:,}")

---
## Cell 8 — Save Output Files

In [ ]:
# Save main output JSON
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
print(f"Saved: {OUTPUT_FILE}")

# Save human-readable log
with open(LOG_FILE, "w", encoding="utf-8") as f:
    f.write("KEYWORD GROUPING LOG\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Original keywords  : {len(data)}\n")
    f.write(f"Output keywords    : {len(result)}\n")
    f.write(f"Keywords merged    : {total_aliases}\n")
    f.write(f"Groups formed      : {total_groups}\n\n")
    f.write("RULE-BASED GROUPS\n" + "-" * 40 + "\n")
    for canon, aliases in sorted(rule_groups.items()):
        f.write(f"  [{canon}]  <-  {aliases}\n")
    f.write("\nSEMANTIC GROUPS\n" + "-" * 40 + "\n")
    for g in semantic_groups:
        aliases = [m for m in g['members'] if m != g['canonical']]
        f.write(f"  [{g['canonical']}]  <-  {aliases}\n")
print(f"Saved: {LOG_FILE}")

print("\nAll done!")

---
## Cell 9 — Preview Results
Browse the groups that were found.

In [ ]:
# Show all keywords that have grouped_with entries
grouped_entries = {k: v for k, v in result.items() if v["grouped_with"]}

print(f"Total grouped keywords: {len(grouped_entries)}\n")
print("Showing first 20 groups:\n")
for canon, entry in list(grouped_entries.items())[:20]:
    print(f"  CANONICAL : {canon}")
    print(f"  MERGED    : {entry['grouped_with']}")
    print(f"  COUNT     : {entry['total_count']}")
    print()

---
## Cell 10 — (Optional) Add More Abbreviations and Re-run Step 1
Found abbreviations that weren't caught? Add them here and re-run from Cell 4.

In [ ]:
# Add your custom abbreviations here
EXTRA_ABBREVS = {
    # "your_abbrev": "full form",
    # "rxn":  "reaction",
    # "temp": "temperature",
}

ABBREV_MAP.update(EXTRA_ABBREVS)
print(f"ABBREV_MAP now has {len(ABBREV_MAP)} entries.")
print("Now re-run Cell 4 onwards to apply the new abbreviations.")